In [ ]:
import os
import sys
import pandas as pd
import  random
import shutil

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling  import RandomOverSampler



# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config


In [ ]:
#  Imports & Configuration

# User parameters

if config.USE_SAMPLED_TRAIN_DATASET:
    INPUT_DIR    = config.SAMPLED_TRAIN_DATASET_DIR         # path of sampled training dataset
else:
    INPUT_DIR    = config.TRAIN_DATASET_DIR                  # path of training dataset

PREPROCESSED_OUTPUT_DIR   = config.PREPROCESSED_DATASET_DIR
TARGET_SIZE  = (640, 640)                          # (height, width)
EXTS         = [".jpg", ".png", ".tif", ".tiff"]   # supported extensions

# CLAHE & denoise settings
CLAHE_CFG    = {"clip_limit": 2.0, "grid_size": (8, 8)}
DENOISE_CFG  = {"h": 10, "template_size": 7, "search_size": 21}

# Ensure output directory exists
os.makedirs(PREPROCESSED_OUTPUT_DIR, exist_ok=True)


In [4]:
data_root    = config.TRAIN_DATASET_HPC_DIR
labels_csv   = config.TRAIN_LABELS_PATH 

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.WORKSPACE_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

box_size    = 32

total_cap     = 2000
neg_cap       = total_cap // 2   # 1000 negatives
pos_cap       = total_cap - neg_cap  # 1000 positives

pipeline = Pipeline([
    ("undersample", RandomUnderSampler(sampling_strategy={0: neg_cap}, random_state=42)),
    ("oversample",  RandomOverSampler (sampling_strategy={1: pos_cap}, random_state=42)),
])

for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id    = row["tomo_id"]
    z          = int(row["Motor axis 0"])
    y          = row["Motor axis 1"]
    x          = row["Motor axis 2"]
    img_width  = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name   = f"slice_{z:04d}.jpg"
    img_path   = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)


# 1) Build a DataFrame of all image paths + binary label
all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

df = pd.DataFrame({
    "path": all_images,
    "label": [1 if p in positive_samples else 0 for p in all_images]
})

print(df)

# 2) Oversample so positives == negatives (1:1 ratio)
X_res, y_res = pipeline.fit_resample(df[["path"]], df["label"])
balanced_paths = X_res["path"].tolist()
balanced_images = X_res["path"].tolist()

print(X_res)
print(y_res)


# 3) Shuffle & split into train/val
random.seed(42)
random.shuffle(balanced_images)
split_idx    = int(len(balanced_images) * 0.8)
train_images = balanced_images[:split_idx]
val_images   = balanced_images[split_idx:]

# 4) Process as before
def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        if not os.path.isfile(img_path):
            print(f"⚠️  Skipping missing image: {img_path}")
            continue
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z       = int(os.path.splitext(os.path.basename(img_path))[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        dst_img = os.path.join(img_out_dir,  f"{base_fn}.jpg")
        dst_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")
        shutil.copy(img_path, dst_img)

        lines = []
        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            cx, cy = x/w, y/h
            bw, bh = (box_size*2)/w, (box_size*2)/h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        with open(dst_lbl, "w") as f:
            f.write("\n".join(lines))

# finally call
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images,   output_images_val,   output_labels_val)


Convert raw sample data to yolo format

[Object Detection dataset overview](https://docs.ultralytics.com/datasets/detect/)

Data augmentation for training data using albumentations liblary.